### Missing values in the data

should they just be removed?

In [82]:
from datetime import datetime
import pandas as pd
import helper_functions_GNN as helper


START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"
df_weather = helper.get_weather_openmeteo(
    lat=58.90,  # central Estonia / Pärnu area
    lon=24.75,
    start=START,
    end=END
)

df_weather


  Fetched 62136 hours: 2019-01-01 00:00:00+00:00 → 2026-02-01 23:00:00+00:00


,temperature,wind_speed_10m,wind_speed_100m
timestamp,,,
2019-01-01 00:00:00+00:00,0.2,7.75,12.59
2019-01-01 01:00:00+00:00,0.9,7.45,12.25
2019-01-01 02:00:00+00:00,1.2,7.29,12.00
2019-01-01 03:00:00+00:00,1.8,7.47,12.32
2019-01-01 04:00:00+00:00,1.9,7.50,12.35
...,...,...,...
2026-02-01 19:00:00+00:00,-20.4,2.90,6.64
2026-02-01 20:00:00+00:00,-20.6,2.57,6.22
2026-02-01 21:00:00+00:00,-21.0,2.34,5.79


In [83]:
#print nans in weather data
print(df_weather.isna().sum())

temperature        0
wind_speed_10m     0
wind_speed_100m    0
dtype: int64


In [84]:
# Check the time resolution of the weather data
diffs = df_weather.index.to_series().diff().dropna().value_counts()
print("Most common time steps:")
print(diffs.head(5))
print(f"\nTotal rows: {len(df_weather)}")
print(f"Date range: {df_weather.index.min()} → {df_weather.index.max()}")
print(f"\nFirst few rows:")
print(df_weather.head())

Most common time steps:
timestamp
0 days 01:00:00    62135
Name: count, dtype: int64

Total rows: 62136
Date range: 2019-01-01 00:00:00+00:00 → 2026-02-01 23:00:00+00:00

First few rows:
                           temperature  wind_speed_10m  wind_speed_100m
timestamp                                                              
2019-01-01 00:00:00+00:00          0.2            7.75            12.59
2019-01-01 01:00:00+00:00          0.9            7.45            12.25
2019-01-01 02:00:00+00:00          1.2            7.29            12.00
2019-01-01 03:00:00+00:00          1.8            7.47            12.32
2019-01-01 04:00:00+00:00          1.9            7.50            12.35


In [77]:
def get_weather_openmeteo(lat: float, lon: float,
                           start: str, end: str) -> pd.DataFrame:
    """
    Free historical weather — no API key, no subscription needed.
    Uses ERA5 reanalysis — covers 1979 to present at hourly resolution.
    """
    r = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude":        lat,
            "longitude":       lon,
            "start_date":      start[:10],  # "2019-01-01"
            "end_date":        end[:10],    # "2026-02-01"
            "hourly":          "temperature_2m,wind_speed_10m,wind_speed_100m",
            "wind_speed_unit": "ms",
            "timezone":        "UTC",
        },
        timeout=60
    )

    if r.status_code != 200:
        print(f"  ERROR {r.status_code}: {r.text[:200]}")
        return pd.DataFrame()

    data = r.json()
    df = pd.DataFrame({
        "timestamp":       pd.to_datetime(data["hourly"]["time"], utc=True),
        "temperature":     data["hourly"]["temperature_2m"],
        "wind_speed_10m":  data["hourly"]["wind_speed_10m"],
        "wind_speed_100m": data["hourly"]["wind_speed_100m"],
    })

    df = df.set_index("timestamp").sort_index()
    print(f"  Fetched {len(df)} hours: {df.index.min()} → {df.index.max()}")
    return df

In [80]:
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"
df_weather = get_weather_openmeteo(
    lat=58.90,  # central Estonia / Pärnu area
    lon=24.75,
    start=START,
    end=END
)

  Fetched 62136 hours: 2019-01-01 00:00:00+00:00 → 2026-02-01 23:00:00+00:00


In [79]:
df_weather.head()

,temperature,wind_speed_10m,wind_speed_100m
timestamp,,,
2026-01-01 00:00:00+00:00,-13.1,1.26,2.26
2026-01-01 01:00:00+00:00,-13.1,0.69,1.81
2026-01-01 02:00:00+00:00,-13.7,1.76,3.71
2026-01-01 03:00:00+00:00,-12.8,1.95,5.26
2026-01-01 04:00:00+00:00,-11.9,2.40,6.14


In [67]:
from datetime import datetime
import pandas as pd
import helper_functions_GNN as helper
from STGNN import STGNN
import numpy as np
import torch
import torch.nn as nn
import torch_geometric.nn as gnn
import matplotlib.pyplot as plt
import sys
import power_scaler as ps

BASE = "https://dashboard.elering.ee/api"
# Fetching data for 2019-2026
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"

df_prices = helper.fetch_all(helper.get_nps_prices, START, END)
df_flows = helper.fetch_all(helper.get_cross_border_flows, START, END)
df_system = helper.fetch_all(helper.get_system_production, START, END)

# Simple — just fetch weather for one central Estonian location - we assume that it is representative for the whole country for now, but we can add more locations later if needed
df_weather = helper.get_weather_history(
    lat=58.90,  # central Estonia / Pärnu area
    lon=24.75,
    start=START,
    end=END
)


# --- 3. standardize to hourly, then daily ---

df_prices_hourly  = df_prices.resample("h").mean() # The data is mostly per 15 minutes and some are hourly
df_flows_hourly   = df_flows.resample("h").mean().fillna(0) # The data is only hourly but resampling to be sure for alignment
df_system_hourly  = df_system.resample("h").mean() # Mostly per 5 minutes, some entries are 0:00 (maybe missing?) and 5 minutes
df_weather_hourly = df_weather.resample("h").mean().reindex(idx, method="ffill")

# --- 4. merge into one wide dataframe ---

df_daily = pd.concat([
    df_prices_hourly.add_prefix("price_"),
    df_flows_hourly.add_prefix("flow_"),
    df_system_hourly.add_prefix("system_"),
    df_weather_hourly.add_prefix("weather_"),
], axis=1).sort_index()



flow_cols = [col for col in df_daily.columns if col.startswith("flow_")]
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)
df_daily = df_daily.dropna(how="all") # Thinking about imputing them


#_h Target: True available energy in Estonia = production + imports
# Scenarios: S1 = full grid, S2 = full isolation, S3 = isolated + wind (TBD)

prices_h = df_prices_hourly.copy()
flows_h  = df_flows_hourly.copy()
system_h = df_system_hourly.copy()
weather_h = df_weather_hourly.copy()

idx      = prices_h.index
flows_h  = flows_h.reindex(idx, method="ffill")
system_h = system_h.reindex(idx, method="ffill")
weather_h = weather_h.reindex(idx, method="ffill")
# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")] 
    - flows_h[("ee", "ru_narva")]
    - flows_h[("ee", "ru_pihkva")]
) # available energy = energy produced in 


print(f"    Hours without enough energy: {(system_h['available_energy'] < 0).sum()}")


# Optional: check what's left
print(f"Shape after cleaning: {df_daily.shape}")
print(f"Remaining missing values:\n{df_daily.isna().sum()[df_daily.isna().sum() > 0]}")

AttributeError: module 'datetime' has no attribute 'fromisoformat'

In [37]:
print(df_daily.columns.tolist())

['price_ee', 'price_fi', 'price_lv', 'price_lt', "flow_('ee', 'fi')", "flow_('ee', 'lv')", "flow_('ee', 'ru_narva')", "flow_('ee', 'ru_pihkva')", 'system_production', 'system_consumption', 'system_losses', 'system_frequency', 'system_system_balance', 'system_ac_balance', 'system_production_renewable', 'system_solar_energy_production']


In [38]:
print("Raw df_flows columns:", df_flows.columns.tolist())
print("After resample columns:", df_flows_hourly.columns.tolist())

Raw df_flows columns: [('ee', 'fi'), ('ee', 'lv'), ('ee', 'ru_narva'), ('ee', 'ru_pihkva')]
After resample columns: [('ee', 'fi'), ('ee', 'lv'), ('ee', 'ru_narva'), ('ee', 'ru_pihkva')]


### Mean, std 

In [39]:



# ==================================================
# ST-GNN: ESTONIAN ENERGY RESILIENCE MODEL
# Target: True energy balance (production + imports - consumption)
# Scenarios: S1 = full grid, S2 = full isolation, S3 = isolated + wind (TBD)
# ==================================================


prices_h = df_prices_hourly.copy()
flows_h  = df_flows_hourly.copy()
system_h = df_system_hourly.copy()

idx      = prices_h.index
flows_h  = flows_h.reindex(idx, method="ffill")
system_h = system_h.reindex(idx, method="ffill")

# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["gross_supply_input"] = (
    system_h["production"]
    - flows_h[("ee", "fi")] #negative flows are imports, positive flows are exports, so we subtract the imports
    - flows_h[("ee", "lv")] - flows_h[("ee", "ru_narva")] - flows_h[("ee", "ru_pihkva")]
)

print(f"    Mean: {system_h['gross_supply_input'].mean():+.1f} MW")
print(f"    Std:  {system_h['gross_supply_input'].std():.1f} MW")
print(f"    Min:  {system_h['gross_supply_input'].min():+.1f} MW")
print(f"    Max:  {system_h['gross_supply_input'].max():+.1f} MW")
print(f"    True deficit hours: {(system_h['gross_supply_input'] < 0).sum()}")


    Mean: +928.7 MW
    Std:  198.0 MW
    Min:  +412.6 MW
    Max:  +1600.1 MW
    True deficit hours: 0


In [40]:
flows_h.columns.tolist()
flows_h

,"(ee, fi)","(ee, lv)","(ee, ru_narva)","(ee, ru_pihkva)"
timestamp,,,,
2019-01-01 00:00:00+00:00,125.7966,119.0860,-165.2681,-10.4020
2019-01-01 01:00:00+00:00,35.8953,157.2594,-118.5069,-0.3477
2019-01-01 02:00:00+00:00,70.3056,122.3419,-135.8722,9.0883
2019-01-01 03:00:00+00:00,60.0575,117.6249,-132.6681,14.6492
2019-01-01 04:00:00+00:00,131.9497,119.3417,-190.1111,0.2314
...,...,...,...,...
2026-01-31 20:00:00+00:00,-999.7225,386.6525,0.0000,0.0000
2026-01-31 21:00:00+00:00,-995.2117,376.0808,0.0000,0.0000
2026-01-31 22:00:00+00:00,-988.6600,305.9417,0.0000,0.0000


EE-FI mean: -624 MW  → Estonia imports ~624 MW from Finland on average

EE-LV mean: +249 MW  → Estonia exports ~249 MW to Latvia on average

In [41]:
flows_h[("ee", "fi")].describe()

count    62113.000000
mean      -624.460111
std        373.506405
min      -1006.658300
25%       -980.483300
50%       -721.183300
75%       -349.475000
max       1033.116000
Name: (ee, fi), dtype: float64

In [42]:
flows_h[("ee", "lv")].describe()

count    62113.000000
mean       248.605089
std        271.094887
min       -820.216700
25%         88.733300
50%        271.600000
75%        441.825000
max        898.666700
Name: (ee, lv), dtype: float64

In [43]:
flows_h[("ee", "ru_narva")].describe()

count    62113.000000
mean        60.788663
std        204.660148
min       -726.750000
25%        -51.583300
50%         33.333300
75%        197.233300
max        901.016700
Name: (ee, ru_narva), dtype: float64

In [44]:
flows_h[("ee", "ru_pihkva")].describe()

count    62113.000000
mean        18.887527
std         51.758048
min       -273.525000
25%          0.000000
50%          0.000000
75%         45.491700
max        339.716700
Name: (ee, ru_pihkva), dtype: float64

In [46]:
prices_h["ee"].describe()

count    62113.000000
mean        88.992090
std         88.443193
min        -60.040000
25%         36.010000
50%         65.120000
75%        113.370000
max       4000.000000
Name: ee, dtype: float64

In [47]:
prices_h["fi"].describe()

count    62113.000000
mean        63.574215
std         80.124744
min       -500.000000
25%         18.220000
50%         40.980000
75%         77.930000
max       1896.000000
Name: fi, dtype: float64

In [48]:
prices_h["lt"].describe()

count    62113.000000
mean        96.029199
std         98.304184
min        -56.550000
25%         38.100000
50%         68.610000
75%        118.840000
max       4000.000000
Name: lt, dtype: float64

In [49]:
prices_h["lv"].describe()

count    62113.000000
mean        95.352533
std         97.050859
min        -56.550000
25%         38.010000
50%         68.440000
75%        118.265000
max       4000.000000
Name: lv, dtype: float64

In [52]:
import requests
import pandas as pd
from dateutil.relativedelta import relativedelta
from datetime import datetime
import torch


BASE = "https://dashboard.elering.ee/api"

def get_nps_prices(start: str, end: str, BASE=BASE) -> pd.DataFrame:

    r = requests.get(BASE + "/nps/price", params={"start": start, "end": end})
    r.raise_for_status()

    data = r.json()["data"]  # {"ee": [...], "lv": [...], "lt": [...], "fi": [...]}

    dfs = []
    for country, records in data.items():
        df = pd.DataFrame(records)
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
        df = df.set_index("timestamp")
        df = df.rename(columns={"price": country})
        dfs.append(df)

    df_prices = pd.concat(dfs, axis=1).sort_index()
    return df_prices

In [53]:
prices = get_nps_prices("2024-01-01T00:00:00.000Z", "2025-01-01T00:00:00.000Z")
prices

,ee,fi,lv,lt
timestamp,,,,
2024-01-01 00:00:00+00:00,28.46,28.46,28.46,28.46
2024-01-01 01:00:00+00:00,26.66,26.66,26.66,26.66
2024-01-01 02:00:00+00:00,24.48,24.48,24.48,24.48
2024-01-01 03:00:00+00:00,24.01,24.01,24.01,24.01
2024-01-01 04:00:00+00:00,21.23,21.23,21.23,21.23
...,...,...,...,...
2024-12-31 20:00:00+00:00,25.08,14.07,25.08,25.08
2024-12-31 21:00:00+00:00,7.94,7.86,7.94,7.94
2024-12-31 22:00:00+00:00,4.01,4.01,4.01,2.05


In [51]:
prices_h

,ee,fi,lv,lt
timestamp,,,,
2019-01-01 00:00:00+00:00,10.0700,10.0700,10.0700,10.0700
2019-01-01 01:00:00+00:00,10.0300,10.0300,10.0300,10.0300
2019-01-01 02:00:00+00:00,4.5600,4.5600,4.5600,4.5600
2019-01-01 03:00:00+00:00,4.8300,4.8300,4.8300,4.8300
2019-01-01 04:00:00+00:00,8.0900,8.0900,8.0900,8.0900
...,...,...,...,...
2026-01-31 20:00:00+00:00,116.9900,116.8925,116.9900,116.9900
2026-01-31 21:00:00+00:00,104.8675,104.7825,104.8675,104.8675
2026-01-31 22:00:00+00:00,89.0050,88.9400,89.0050,89.0050


## cosine and sine

In [ ]:

# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")] 
    - flows_h[("ee", "ru_narva")] #also russia since it was present before 2025
    - flows_h[("ee", "ru_pihkva")]
) # available energy = energy produced in 


print(f" Hours without enough energy (or energy =0): {(system_h['available_energy'] <= 0).sum()}")



# Calendar features (sine/cosine encoding — avoids treating Mon=1, Sun=7 as numeric)
hour_sin  = np.sin(2 * np.pi * idx.hour / 24)
hour_cos  = np.cos(2 * np.pi * idx.hour / 24)
dow_sin   = np.sin(2 * np.pi * idx.dayofweek / 7)
dow_cos   = np.cos(2 * np.pi * idx.dayofweek / 7)
month_sin = np.sin(2 * np.pi * (idx.month - 1) / 12) # Range is originally from 1 to 12, so subtract 1 to get 0-11 for encoding
month_cos = np.cos(2 * np.pi * (idx.month - 1) / 12) #to be consistent with month_sin and dayofweek encoding

# Frequency deviation from 50 Hz — real-time grid stress signal -- research later
freq_deviation = (system_h["frequency"] - 50.0).fillna(0) # Check later on missing values 



 Hours without enough energy (or energy =0): 0


In [ ]:
month_sin.value_counts()

timestamp
 5.000000e-01    9793
 0.000000e+00    5952
 8.660254e-01    5208
 8.660254e-01    5208
 1.224647e-16    5208
-5.000000e-01    5208
-1.000000e+00    5208
-5.000000e-01    5208
 1.000000e+00    5040
-8.660254e-01    5040
-8.660254e-01    5040
Name: count, dtype: int64

In [ ]:
idx.dayofweek.unique()

Index([1, 2, 3, 4, 5, 6, 0], dtype='int32', name='timestamp')

In [ ]:
# Frequency deviation from 50 Hz — real-time grid stress signal -- research later
freq_deviation = (system_h["frequency"] - 50.0).fillna(0) # Check later on missing values 


In [ ]:
freq_deviation 

timestamp
2019-01-01 00:00:00+00:00    0.000458
2019-01-01 01:00:00+00:00    0.000550
2019-01-01 02:00:00+00:00    0.001867
2019-01-01 03:00:00+00:00    0.000158
2019-01-01 04:00:00+00:00    0.002608
                               ...   
2026-01-31 20:00:00+00:00    0.005000
2026-01-31 21:00:00+00:00   -0.005000
2026-01-31 22:00:00+00:00    0.010000
2026-01-31 23:00:00+00:00   -0.006667
2026-02-01 00:00:00+00:00   -0.040000
Freq: h, Name: frequency, Length: 62113, dtype: float64

In [63]:
from datetime import datetime
import pandas as pd
import helper_functions_GNN as helper
from STGNN import STGNN
import numpy as np
import torch
import torch.nn as nn
import torch_geometric.nn as gnn
import matplotlib.pyplot as plt
import sys
#import power_scaler as ps

BASE = "https://dashboard.elering.ee/api"
# Fetching data for 2019-2026
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"

df_prices = helper.fetch_all(helper.get_nps_prices, START, END)
df_flows = helper.fetch_all(helper.get_cross_border_flows, START, END)
df_system = helper.fetch_all(helper.get_system_production, START, END)

# --- 3. standardize to hourly, then daily ---

df_prices_hourly  = df_prices.resample("h").mean() # The data is mostly per 15 minutes and some are hourly
df_flows_hourly   = df_flows.resample("h").mean().fillna(0) # The data is only hourly but resampling to be sure for alignment
df_system_hourly  = df_system.resample("h").mean() # Mostly per 5 minutes, some entries are 0:00 (maybe missing?) and 5 minutes

# --- 4. merge into one wide dataframe ---

df_daily = pd.concat([
    df_prices_hourly.add_prefix("price_"),
    df_flows_hourly.add_prefix("flow_"),
    df_system_hourly.add_prefix("system_"),
], axis=1).sort_index()



flow_cols = [col for col in df_daily.columns if col.startswith("flow_")]
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)
df_daily = df_daily.dropna(how="all") # Thinking about imputing them


# Target: True available energy in Estonia = production + imports
# Scenarios: S1 = full grid, S2 = full isolation, S3 = isolated + wind (TBD)

prices_h = df_prices_hourly.copy()
flows_h  = df_flows_hourly.copy()
system_h = df_system_hourly.copy()

idx      = prices_h.index
flows_h  = flows_h.reindex(idx, method="ffill")
system_h = system_h.reindex(idx, method="ffill")

# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")] 
    - flows_h[("ee", "ru_narva")]
    - flows_h[("ee", "ru_pihkva")]
) # available energy = energy produced in 


print(f"    Hours without enough energy: {(system_h['available_energy'] < 0).sum()}")



# Calendar features (sine/cosine encoding — avoids treating Mon=1, Sun=7 as numeric)
hour_sin  = np.sin(2 * np.pi * idx.hour / 24)
hour_cos  = np.cos(2 * np.pi * idx.hour / 24)
dow_sin   = np.sin(2 * np.pi * idx.dayofweek / 7)
dow_cos   = np.cos(2 * np.pi * idx.dayofweek / 7)
month_sin = np.sin(2 * np.pi * (idx.month - 1) / 12) # Range is originally from 1 to 12, so subtract 1 to get 0-11 for encoding
month_cos = np.cos(2 * np.pi * (idx.month - 1) / 12)

# Frequency deviation from 50 Hz — real-time grid stress signal -- research later

essential_cols = [ "available_energy", "production", "production_renewable", "frequency" ]

#use interpolation + forward/backward fill to handle missing values in essential features, to avoid losing hours of data. We can experiment with more sophisticated imputation later if needed, but this should be sufficient for now given the relatively low frequency of missing values.
for col in essential_cols:
    if col in system_h.columns:
        system_h[col] = system_h[col].interpolate(method='linear')
        system_h[col] = system_h[col].ffill().bfill()

# Frequency deviation from 50 Hz — real-time grid stress signal -- research later
freq_deviation = (system_h["frequency"] - 50.0).fillna(0) # Check later on missing values 
# https://clouglobal.com/power-grid-frequency-why-is-it-important/ - source about frequency deviation and its importance as a grid stress signal. We can research more about it later and maybe add it as a feature if we find it useful for the model. For now, we include it as a potential signal of grid stress that could correlate with supply-demand imbalances.



# EE node: full feature set
ee_feats = pd.DataFrame({
    "available_energy":   system_h["available_energy"],
    "production_renewable": system_h["production_renewable"],
    "production":           system_h["production"],
    #"consumption":          system_h["consumption"],
    "flow_fi":              flows_h[("ee", "fi")],
    "flow_lv":              flows_h[("ee", "lv")],
    "price":                prices_h["ee"],
    "freq_deviation":       freq_deviation,
    "hour_sin":             hour_sin,
    "hour_cos":             hour_cos,
    "dow_sin":              dow_sin,
    "dow_cos":              dow_cos,
    "month_sin":            month_sin,
    "month_cos":            month_cos,
}, index=idx).fillna(0) # Might change it into imputing values later instead of filling with 0, but for now it is fine since we have no missing values in these features

# Neighbouring nodes — only features we actually have for them - we only got prices for fi, lv, lt,
fi_feats = pd.DataFrame({
    "price":   prices_h["fi"],
    "flow_fi": flows_h[("ee", "fi")],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

lv_feats = pd.DataFrame({
    "price":   prices_h["lv"],
    "flow_lv": flows_h[("ee", "lv")],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

lt_feats = pd.DataFrame({
    "price": prices_h["lt"],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)


# Stack into (T, N, F)
node_data = np.stack([
    ee_feats.values,   # node 0: EE
    fi_feats.values,   # node 1: FI
    lv_feats.values,   # node 2: LV
    lt_feats.values,   # node 3: LT
], axis=1)

# We didn't include Russia since we don't have good features for it

NUM_NODES     = 4
NUM_FEATURES  = node_data.shape[2]
FEATURE_NAMES = list(ee_feats.columns)
SUPPLY_IDX = FEATURE_NAMES.index("available_energy")  # target is the (production + imports)
FLOW_FI_IDX   = FEATURE_NAMES.index("flow_fi")
FLOW_LV_IDX   = FEATURE_NAMES.index("flow_lv")
RENEW_IDX     = FEATURE_NAMES.index("production_renewable")
PROD_IDX      = FEATURE_NAMES.index("production")



# ==================================================
# STEP 2: SEQUENCES + SPLITS
# ==================================================
print("\n[2/6] Creating sequences...")

#  predicting what the energy balance will be 24 hours from now, given the last 48 hours. 
SEQ_LEN = 48   # 2 days of hourly history - thinking about including more 
HORIZON = 24   # predict 24h ahead

y_target_values = system_h["available_energy"].values 

X_list, y_list = [], []
for t in range(SEQ_LEN, len(node_data) - HORIZON): #range starts at SEQ_LEN to ensure we have enough history for the first sample, and ends at len(node_data) - HORIZON to ensure we have a valid target value for the last sample
    X_list.append(node_data[t - SEQ_LEN:t]) # Each sample is a (SEQ_LEN, N, F) tensor covering hours t-SEQ_LEN to t-1
    y_list.append(y_target_values[t + HORIZON - 1])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

print(f"Total samples: {len(X)}")
print(f"Sample shape (T, N, F): {X[0].shape}")


/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = combined[~combined.index.duplicated(keep="first")]
/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = combined[~combined.index.duplicated(keep="first

    Hours without enough energy: 0

[2/6] Creating sequences...
Total samples: 62041
Sample shape (T, N, F): (48, 4, 13)


In [64]:

# January 2026 = held-out test (never seen during training)
jan2026_start = np.searchsorted(
    idx[SEQ_LEN:-HORIZON],
    pd.Timestamp("2026-01-01", tz="UTC")
)

X_train_full, y_train_full = X[:jan2026_start], y[:jan2026_start] #before January 2026 for training (including a validation split from this set)
X_test,        y_test       = X[jan2026_start:], y[jan2026_start:] # Jaendnuary 2026 for testing (final evaluation on unseen data)
 # January 2026 for testing (final evaluation on unseen data)

val_split = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full[:val_split],  y_train_full[:val_split]
X_val,   y_val   = X_train_full[val_split:],  y_train_full[val_split:]

print(f"Train: {len(X_train):,} ({X_train.shape}) | Val: {len(X_val):,} | Test (Jan 2026): {len(X_test):,}")



Train: 49,056 ((49056, 48, 4, 13)) | Val: 12,264 | Test (Jan 2026): 721


In [66]:
import power_scaler as ps

ModuleNotFoundError: No module named 'power_scaler'

In [ ]:
node_data.shape = (T, N, F)
                   ↑  ↑  ↑
                   |  |  └── F = Features = 13
                   |  |       (available_energy, production, flow_fi, ...)
                   |  |
                   |  └───── N = Nodes = 4
                   |          (EE, FI, LV, LT)
                   |
                   └──────── T = Timesteps = ~62,000
                              (one row per hour, 2019-2026)

t starts at t=48

node_data.shape = (T, N, F)
                   ↑  ↑  ↑
                   |  |  └── F = Features = 13
                   |  |       (available_energy, production, flow_fi, ...)
                   |  |
                   |  └───── N = Nodes = 4
                   |          (EE, FI, LV, LT)
                   |
                   └──────── T = Timesteps = ~62,000
                              (one row per hour, 2019-2026)

In [56]:
X = [1,2,4,5,7,7,4]
X[:3]

[1, 2, 4]

### check with weather data

In [87]:
from datetime import datetime
import pandas as pd
import helper_functions_GNN as helper
from STGNN import STGNN
import numpy as np
import torch
import torch.nn as nn
import torch_geometric.nn as gnn
import matplotlib.pyplot as plt
import sys
import power_scaler as ps
#import  GNN_optimzer as opt

BASE = "https://dashboard.elering.ee/api"
# Fetching data for 2019-2026
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"

df_prices = helper.fetch_all(helper.get_nps_prices, START, END)
df_flows = helper.fetch_all(helper.get_cross_border_flows, START, END)
df_system = helper.fetch_all(helper.get_system_production, START, END)

# Simple — just fetch weather for one central Estonian location - we assume that it is representative for the whole country for now, but we can add more locations later if needed
df_weather = helper.get_weather_openmeteo(
    lat=58.90,  # central Estonia / Pärnu area
    lon=24.75,
    start=START,
    end=END
)


# --- 3. standardize to hourly, then daily ---

df_prices_hourly  = df_prices.resample("h").mean() # The data is mostly per 15 minutes and some are hourly
df_flows_hourly   = df_flows.resample("h").mean().fillna(0) # The data is only hourly but resampling to be sure for alignment
df_system_hourly  = df_system.resample("h").mean() # Mostly per 5 minutes, some entries are 0:00 (maybe missing?) and 5 minutes
df_weather_hourly = df_weather.resample("h").mean()

# --- 4. merge into one wide dataframe ---

df_daily = pd.concat([
    df_prices_hourly.add_prefix("price_"),
    df_flows_hourly.add_prefix("flow_"),
    df_system_hourly.add_prefix("system_"),
    df_weather_hourly.add_prefix("weather_"),
], axis=1).sort_index()



flow_cols = [col for col in df_daily.columns if col.startswith("flow_")]
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)
df_daily = df_daily.dropna(how="all") # Thinking about imputing them


#_h Target: True available energy in Estonia = production + imports
# Scenarios: S1 = full grid, S2 = full isolation, S3 = isolated + wind (TBD)

prices_h = df_prices_hourly.copy()
flows_h  = df_flows_hourly.copy()
system_h = df_system_hourly.copy()
weather_h = df_weather_hourly.copy()

idx       = prices_h.index
flows_h   = flows_h.reindex(idx, method="ffill")
system_h  = system_h.reindex(idx, method="ffill")
weather_h = df_weather_hourly.reindex(idx, method="ffill")  # reindex here — idx now defined
# True energy balance: production + all imports - consumption
# Positive = surplus, Negative = real deficit even after imports
system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")] 
    - flows_h[("ee", "ru_narva")]
    - flows_h[("ee", "ru_pihkva")]
) # available energy = energy produced in 


print(f"    Hours without enough energy: {(system_h['available_energy'] < 0).sum()}")



# Calendar features (sine/cosine encoding — avoids treating Mon=1, Sun=7 as numeric)
hour_sin  = np.sin(2 * np.pi * idx.hour / 24)
hour_cos  = np.cos(2 * np.pi * idx.hour / 24)
dow_sin   = np.sin(2 * np.pi * idx.dayofweek / 7)
dow_cos   = np.cos(2 * np.pi * idx.dayofweek / 7)
month_sin = np.sin(2 * np.pi * (idx.month - 1) / 12) # Range is originally from 1 to 12, so subtract 1 to get 0-11 for encoding
month_cos = np.cos(2 * np.pi * (idx.month - 1) / 12)

# Frequency deviation from 50 Hz — real-time grid stress signal -- research laterhttps://clouglobal.com/power-grid-frequency-why-is-it-important/

essential_cols = [ "available_energy", "production", "production_renewable", "frequency" ]

for col in essential_cols:
    if col in system_h.columns:
        system_h[col] = system_h[col].interpolate(method='linear')
        system_h[col] = system_h[col].ffill().bfill()

freq_deviation = (system_h["frequency"] - 50.0)


# EE node: full feature set
ee_feats = pd.DataFrame({
    "available_energy":     system_h["available_energy"],
    "production_renewable": system_h["production_renewable"],
    "production":           system_h["production"],
    "flow_fi":              flows_h[("ee", "fi")],
    "flow_lv":              flows_h[("ee", "lv")],
    "price":                prices_h["ee"],
    "temperature":          weather_h["temperature"],    # ← now active
    "wind_speed_10m":       weather_h["wind_speed_10m"], # ← now active
    "freq_deviation":       freq_deviation,
    "hour_sin":             hour_sin,
    "hour_cos":             hour_cos,
    "dow_sin":              dow_sin,
    "dow_cos":              dow_cos,
    "month_sin":            month_sin,
    "month_cos":            month_cos,
}, index=idx).fillna(0)# Might change it into imputing values later instead of filling with 0, but for now it is fine since we have no missing values in these features

# Neighbouring nodes — only features we actually have for them
fi_feats = pd.DataFrame({
    "price":   prices_h["fi"],
    "flow_fi": flows_h[("ee", "fi")],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

lv_feats = pd.DataFrame({
    "price":   prices_h["lv"],
    "flow_lv": flows_h[("ee", "lv")],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

lt_feats = pd.DataFrame({
    "price": prices_h["lt"],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

# Stack into (T, N, F)
node_data = np.stack([
    ee_feats.values,   # node 0: EE
    fi_feats.values,   # node 1: FI
    lv_feats.values,   # node 2: LV
    lt_feats.values,   # node 3: LT
], axis=1)

# We didn't include Russia since we don't have good features for it

NUM_NODES     = 4
NUM_FEATURES  = node_data.shape[2]
FEATURE_NAMES = list(ee_feats.columns)
SUPPLY_IDX  = FEATURE_NAMES.index("available_energy")
FLOW_FI_IDX = FEATURE_NAMES.index("flow_fi")
FLOW_LV_IDX = FEATURE_NAMES.index("flow_lv")
RENEW_IDX   = FEATURE_NAMES.index("production_renewable")
PROD_IDX    = FEATURE_NAMES.index("production")
TEMP_IDX    = FEATURE_NAMES.index("temperature")      # ← new
WIND_IDX    = FEATURE_NAMES.index("wind_speed_10m")   # ← new



/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()
/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()


  Fetched 62136 hours: 2019-01-01 00:00:00+00:00 → 2026-02-01 23:00:00+00:00
    Hours without enough energy: 0


In [89]:
print(f"Weather features added: {['temperature', 'wind_speed_10m']}")
print(weather_h[["temperature", "wind_speed_10m"]].describe())


Weather features added: ['temperature', 'wind_speed_10m']
        temperature  wind_speed_10m
count  62113.000000    62113.000000
mean       7.005237        3.724546
std        8.820231        1.684441
min      -25.800000        0.000000
25%        0.600000        2.460000
50%        6.300000        3.520000
75%       13.900000        4.800000
max       31.000000       12.960000


In [90]:
flows_h.columns.tolist()

[('ee', 'fi'), ('ee', 'lv'), ('ee', 'ru_narva'), ('ee', 'ru_pihkva')]

In [94]:
#print nans in flows_h
print(flows_h.isna().sum())
#print how many is 0
print((flows_h == 0).sum())

(ee, fi)           0
(ee, lv)           0
(ee, ru_narva)     0
(ee, ru_pihkva)    0
dtype: int64
(ee, fi)              37
(ee, lv)               0
(ee, ru_narva)      8587
(ee, ru_pihkva)    17386
dtype: int64


flow_h.describe

In [95]:
entsoe = pd.read_csv(
    "../data/entsoe_production_hourly.csv", index_col=0, parse_dates=True
)
wind_mw = entsoe["wind_onshore"].reindex(idx, method="ffill").fillna(0)
print(f"    wind_mw NaN count after reindex: {wind_mw.isna().sum()}")


print(f"    wind_mw stats:\n{wind_mw.describe()}")


    wind_mw NaN count after reindex: 0
    wind_mw stats:
count    62113.000000
mean       101.618914
std         90.085783
min          0.000000
25%         35.100000
50%         73.100000
75%        146.000000
max        588.400000
Name: wind_onshore, dtype: float64


In [97]:
entsoe.columns.tolist()

['biomass',
 'fossil_coal-derived_gas',
 'fossil_gas',
 'fossil_oil_shale',
 'fossil_peat',
 'hydro_run-of-river_and_pondage',
 'other',
 'other_renewable',
 'solar',
 'waste',
 'wind_onshore',
 'capacity_factor',
 'installed_mw']

In [98]:
wind_mw.head()

timestamp
2019-01-01 00:00:00+00:00    236.7
2019-01-01 01:00:00+00:00    227.0
2019-01-01 02:00:00+00:00    206.4
2019-01-01 03:00:00+00:00    201.5
2019-01-01 04:00:00+00:00    198.6
Freq: h, Name: wind_onshore, dtype: float64

In [100]:
#print latest date for wind_mw
print(f"    Latest date in wind_mw: {wind_mw.index.max()}")

    Latest date in wind_mw: 2026-02-01 00:00:00+00:00


In [1]:
from datetime import datetime
import pandas as pd
from sympy import isolate
import helper_functions_GNN as helper
from STGNN import STGNN
import numpy as np
import torch
import torch.nn as nn
import torch_geometric.nn as gnn
import matplotlib.pyplot as plt
import sys
import power_scaler as ps
import  GNN_optimizer as opt
import os

BASE = "https://dashboard.elering.ee/api"
# Fetching data for 2019-2026
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"


df_prices = helper.fetch_all(helper.get_nps_prices, START, END)
df_flows = helper.fetch_all(helper.get_cross_border_flows, START, END)
df_system = helper.fetch_all(helper.get_system_production, START, END)

# Simple — just fetch weather for one central Estonian location - we assume that it is representative for the whole country for now, but we can add more locations later if needed
df_weather = helper.get_weather_openmeteo(
    lat=58.90,  # central Estonia / Pärnu area
    lon=24.75,
    start=START,
    end=END
)


# --- 3. standardize to hourly, then daily ---

df_prices_hourly  = df_prices.resample("h").mean() # The data is mostly per 15 minutes and some are hourly
df_flows_hourly   = df_flows.resample("h").mean().fillna(0) # The data is only hourly but resampling to be sure for alignment
df_system_hourly  = df_system.resample("h").mean() # Mostly per 5 minutes, some entries are 0:00 (maybe missing?) and 5 minutes
df_weather_hourly = df_weather.resample("h").mean()

# --- 4. merge into one wide dataframe ---

df_daily = pd.concat([
    df_prices_hourly.add_prefix("price_"),
    df_flows_hourly.add_prefix("flow_"),
    df_system_hourly.add_prefix("system_"),
    df_weather_hourly.add_prefix("weather_"),
], axis=1).sort_index()



flow_cols = [col for col in df_daily.columns if col.startswith("flow_")]
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)
df_daily = df_daily.dropna(how="all")


#_h Target: Domestic EE production
# Scenarios: S1 = full grid with historical flows, S2-S4 = full isolation (flows=0) + wind investments

prices_h = df_prices_hourly.copy()
flows_h  = df_flows_hourly.copy()
system_h = df_system_hourly.copy()
weather_h = df_weather_hourly.copy()

idx       = prices_h.index
flows_h   = flows_h.reindex(idx, method="ffill")
system_h  = system_h.reindex(idx, method="ffill")
weather_h = df_weather_hourly.reindex(idx, method="ffill")  # reindex here — idx now defined

system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")]
    - flows_h[("ee", "ru_narva")]
    - flows_h[("ee", "ru_pihkva")]
)

print(f"    Hours with production deficit (without imports): {(system_h['available_energy'] < 0).sum()}")



# Calendar features (sine/cosine encoding — avoids treating Mon=1, Sun=7 as numeric)
hour_sin  = np.sin(2 * np.pi * idx.hour / 24)
hour_cos  = np.cos(2 * np.pi * idx.hour / 24)
dow_sin   = np.sin(2 * np.pi * idx.dayofweek / 7)
dow_cos   = np.cos(2 * np.pi * idx.dayofweek / 7)
month_sin = np.sin(2 * np.pi * (idx.month - 1) / 12)
month_cos = np.cos(2 * np.pi * (idx.month - 1) / 12)

essential_cols = [ "available_energy", "production", "production_renewable", "frequency" ]

for col in essential_cols:
    if col in system_h.columns:
        system_h[col] = system_h[col].interpolate(method='linear')
        system_h[col] = system_h[col].ffill().bfill()

freq_deviation = (system_h["frequency"] - 50.0)


# Define horizon early so we can use it for the lagged feature below
HORIZON = 24   # predict 24h ahead (also defined again in STEP 2 — consistent)

# ── Flow sign diagnostic ─────────────────────────────────────────────────────
# The Elering API convention: positive = EE exports, negative = EE imports.
print("[DIAGNOSTIC] Mean flow EE→FI (should be negative if EE is net importer from FI):",
      flows_h[("ee", "fi")].mean().round(1))
print("[DIAGNOSTIC] Mean flow EE→LV (should be negative if EE imports from LV):",
      flows_h[("ee", "lv")].mean().round(1))
# ─────────────────────────────────────────────────────────────────────────────

entsoe = pd.read_csv(
    "../data/entsoe_production_hourly.csv", index_col=0, parse_dates=True
)
wind_mw = entsoe["wind_onshore"].reindex(idx, method="ffill").fillna(0)
print(f"    wind_mw NaN count after reindex: {wind_mw.isna().sum()}")

# EE node: full feature set
# Drivers of production: renewable components, weather, calendar, market context (flows/prices)
# Autoregressive: lagged production captures temporal inertia and daily patterns
# Target: production (not included as feature to avoid leakage)
ee_feats = pd.DataFrame({
    "production_lag1": system_h["production"].shift(1),
    "production_lag24": system_h["production"].shift(24),
    "available_energy_lag24": system_h["available_energy"].shift(HORIZON),
    "available_energy":     system_h["available_energy"],
    "production_renewable": system_h["production_renewable"], # der kommeer ikke data leakage ved at bruge denne
    "wind_mw":              wind_mw,
    "flow_fi":              flows_h[("ee", "fi")],
    "flow_lv":              flows_h[("ee", "lv")],
    "price":                prices_h["ee"],
    "temperature":          weather_h["temperature"],
    "wind_speed_10m":       weather_h["wind_speed_10m"],
    "freq_deviation":       freq_deviation,
    "hour_sin":             hour_sin,
    "hour_cos":             hour_cos,
    "dow_sin":              dow_sin,
    "dow_cos":              dow_cos,
    "month_sin":            month_sin,
    "month_cos":            month_cos,
}, index=idx).fillna(0)

# Neighbouring nodes — only features we actually have for them
fi_feats = pd.DataFrame({
    "price":   prices_h["fi"],
    "flow_fi": flows_h[("ee", "fi")],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

lv_feats = pd.DataFrame({
    "price":   prices_h["lv"],
    "flow_lv": flows_h[("ee", "lv")],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

lt_feats = pd.DataFrame({
    "price": prices_h["lt"],
}, index=idx).reindex(columns=ee_feats.columns, fill_value=0).fillna(0)

# Stack into (T, N, F)
node_data = np.stack([
    ee_feats.values,   # node 0: EE
    fi_feats.values,   # node 1: FI
    lv_feats.values,   # node 2: LV
    lt_feats.values,   # node 3: LT
], axis=1)

NUM_NODES     = 4
NUM_FEATURES  = node_data.shape[2]
FEATURE_NAMES = list(ee_feats.columns)

# Features der bruges i scenariosimulering:
PROD_LAG1_IDX  = FEATURE_NAMES.index("production_lag1")
PROD_LAG24_IDX = FEATURE_NAMES.index("production_lag24")
RENEW_IDX      = FEATURE_NAMES.index("production_renewable")  # ← nu virker det
WIND_MW_IDX    = FEATURE_NAMES.index("wind_mw")
FLOW_FI_IDX    = FEATURE_NAMES.index("flow_fi")
FLOW_LV_IDX    = FEATURE_NAMES.index("flow_lv")
TEMP_IDX       = FEATURE_NAMES.index("temperature")
WIND_IDX       = FEATURE_NAMES.index("wind_speed_10m")

# available_energy er stadig en feature (lagged version) — ikke target
AVAIL_LAG_IDX  = FEATURE_NAMES.index("available_energy_lag24")
AVAIL_IDX      = FEATURE_NAMES.index("available_energy")
print(f"    Features ({NUM_FEATURES}): {FEATURE_NAMES}")


# ==================================================
# STEP 2: SEQUENCES + SPLITS
# ==================================================
print("\n[2/6] Creating sequences...")

SEQ_LEN = 48   # 2 days of hourly history

y_target_values = system_h["production"].values #production as target

X_list, y_list = [], []
for t in range(SEQ_LEN, len(node_data) - HORIZON):
    X_list.append(node_data[t - SEQ_LEN:t])
    y_list.append(y_target_values[t + HORIZON - 1])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)

# January 2026 = held-out test (never seen during training)
jan2026_start = np.searchsorted(
    idx[SEQ_LEN:-HORIZON],
    pd.Timestamp("2026-01-01", tz="UTC")
)
X_train_full, y_train_full = X[:jan2026_start], y[:jan2026_start]
X_test,        y_test       = X[jan2026_start:], y[jan2026_start:]

val_split = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full[:val_split],  y_train_full[:val_split]
X_val,   y_val   = X_train_full[val_split:],  y_train_full[val_split:]

print(f"Train: {len(X_train):,} ({X_train.shape}) | Val: {len(X_val):,} | Test (Jan 2026): {len(X_test):,}")



/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()
/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/project/helper_functions_GNN.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(chunks).sort_index()


  Fetched 62136 hours: 2019-01-01 00:00:00+00:00 → 2026-02-01 23:00:00+00:00
    Hours with production deficit (without imports): 0
[DIAGNOSTIC] Mean flow EE→FI (should be negative if EE is net importer from FI): -624.5
[DIAGNOSTIC] Mean flow EE→LV (should be negative if EE imports from LV): 248.6
    wind_mw NaN count after reindex: 0
    Features (18): ['production_lag1', 'production_lag24', 'available_energy_lag24', 'available_energy', 'production_renewable', 'wind_mw', 'flow_fi', 'flow_lv', 'price', 'temperature', 'wind_speed_10m', 'freq_deviation', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']

[2/6] Creating sequences...
Train: 49,056 ((49056, 48, 4, 18)) | Val: 12,264 | Test (Jan 2026): 721


In [3]:
import pandas as pd

wind_scenarios = pd.read_csv(
    "/Users/sandrarune/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/8. sem/Advanced BA/Advanced_BA/data/wind_production_scenarios.csv",
    index_col=0, 
    parse_dates=True
)

print(wind_scenarios.shape)
print(wind_scenarios.columns.tolist())
print(wind_scenarios.head())
print(wind_scenarios.index[:3])
print(wind_scenarios.index[-3:])

(744, 3)
['wind_mwh_baseline', 'wind_mwh_scenA', 'wind_mwh_scenB']
                           wind_mwh_baseline  wind_mwh_scenA  wind_mwh_scenB
timestamp                                                                   
2026-01-01 00:00:00+00:00           0.000000       17.316040       36.260532
2026-01-01 01:00:00+00:00           0.000000       25.743496       54.969447
2026-01-01 02:00:00+00:00           4.662937       39.321736       84.264279
2026-01-01 03:00:00+00:00          48.182508      101.267046      177.841431
2026-01-01 04:00:00+00:00          88.023939      155.354563      239.602512
DatetimeIndex(['2026-01-01 00:00:00+00:00', '2026-01-01 01:00:00+00:00',
               '2026-01-01 02:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='timestamp', freq=None)
DatetimeIndex(['2026-01-31 21:00:00+00:00', '2026-01-31 22:00:00+00:00',
               '2026-01-31 23:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='timestamp', freq=None)


In [4]:
wind_scenA_raw = (
    wind_scenarios["wind_mwh_scenA"] - wind_scenarios["wind_mwh_baseline"]
).values

wind_scenB_raw = (
    wind_scenarios["wind_mwh_scenB"] - wind_scenarios["wind_mwh_baseline"]
).values

# Pad med SEQ_LEN ekstra værdier så alle test samples får vind
SEQ_LEN = 48
pad_A = np.full(SEQ_LEN, wind_scenA_raw[-1])  # gentag sidste værdi
pad_B = np.full(SEQ_LEN, wind_scenB_raw[-1])

wind_scenA = np.concatenate([wind_scenA_raw, pad_A])
wind_scenB = np.concatenate([wind_scenB_raw, pad_B])

print(f"wind_scenA length: {len(wind_scenA)}  (needed: {len(X_test) + SEQ_LEN})")
print(f"wind_scenB length: {len(wind_scenB)}")


wind_scenA length: 792  (needed: 769)
wind_scenB length: 792


In [6]:
print("Wind scenario values (MW additional production):")
print(f"Baseline mean: {wind_scenarios['wind_mwh_baseline'].mean():.1f} MW")
print(f"  Scenario A mean: {wind_scenA[:744].mean():.1f} MW")
print(f"  Scenario A max:  {wind_scenA[:744].max():.1f} MW")
print(f"  Scenario A min:  {wind_scenA[:744].min():.1f} MW")
print(f"\n  Scenario B mean: {wind_scenB[:744].mean():.1f} MW")
print(f"  Scenario B max:  {wind_scenB[:744].max():.1f} MW")
print(f"  Scenario B min:  {wind_scenB[:744].min():.1f} MW")

Wind scenario values (MW additional production):
Baseline mean: 105.3 MW
  Scenario A mean: 67.8 MW
  Scenario A max:  297.4 MW
  Scenario A min:  0.0 MW

  Scenario B mean: 171.0 MW
  Scenario B max:  784.4 MW
  Scenario B min:  0.0 MW
